# Enterprise RAG Quality Evaluation Pipeline

An interactive, end-to-end Jupyter Notebook for testing, benchmarking, and tracking the retrieval and generation quality of the LangGraph documentation chatbot.

### Pipeline Stages:
1. **Setup & Client Initialization**: Load environment configuration and configure OpenRouter RAGAS wrappers.
2. **Step 1: Testset Generation (Optional)**: Synthesize multi-hop evaluation questions from raw document chunks via Knowledge Graph.
3. **Step 2: Inspect Evaluation Dataset**: Review synthesized question distributions, references, and ground truth contexts in LangSmith schema.
4. **Step 3: Query LangGraph Agent**: Execute evaluation queries directly via in-process `agent_graph.ainvoke()` (or via LangGraph Server HTTP API port 2024).
5. **Step 4: Compute RAGAS Metrics**: Score Faithfulness, Answer Relevancy, Context Precision, and Context Recall.
6. **Step 5: Historical Evolution & Visualization**: Analyze benchmark progression across historical CSV logs with threshold charts.
7. **Step 6: Sync with Evaluation Journal**: Automatically format and append benchmark summaries to `JOURNAL.md`.
8. **Step 7: Upload Dataset to LangSmith**: Ingest the evaluation test set directly into LangSmith Dataset & Observability Hub.
9. **Step 8: All-in-One Automated Benchmark Execution**: Single-call end-to-end pipeline execution.

In [ ]:
# region Setup
import asyncio
import glob
import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)
from ragas.run_config import RunConfig

# Path resolution: ensure project root is in sys.path
CURRENT_DIR = Path.cwd() if Path.cwd().name == "eval" else Path.cwd() / "dataset" / "eval"
ROOT_DIR = CURRENT_DIR.parents[1]
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

load_dotenv(ROOT_DIR / ".env")

print(f"[Setup-init] Current Directory: {CURRENT_DIR}")
print(f"[Setup-init] Root Directory:    {ROOT_DIR}")
print(f"[Setup-init] OpenRouter Key:    {'Set' if os.getenv('OPENROUTER_API_KEY') else 'Missing'}")
print(f"[Setup-init] LangSmith Key:     {'Set' if os.getenv('LANGSMITH_API_KEY') else 'Missing'}")


def get_eval_models(temperature: float = 0.0, is_generator: bool = False):
    """Initializes LLM and Embeddings wrappers for Ragas evaluation and testset generation."""
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("[EvalClient-init] OPENROUTER_API_KEY is not set in environment.")

    base_url = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
    model = os.getenv("OPENROUTER_MODEL", "deepseek/deepseek-v4-flash-0731")
    embed_model = os.getenv("OPENROUTER_EMBED_MODEL", "nvidia/nemotron-3-embed-1b:free")
    provider_sort = os.getenv("OPENROUTER_PROVIDER_SORT", "throughput")
    provider_ignore_raw = os.getenv("OPENROUTER_PROVIDER_IGNORE", "wafer")
    provider_ignore = [p.strip() for p in provider_ignore_raw.split(",") if p.strip()]

    provider_config = {"allow_fallbacks": True}
    if provider_sort:
        provider_config["sort"] = provider_sort
    if provider_ignore:
        provider_config["ignore"] = provider_ignore

    extra_body = {
        "provider": provider_config,
        "reasoning": {
            "effort": "medium",
        },
    }

    llm = ChatOpenAI(
        model=model,
        api_key=api_key,
        base_url=base_url,
        temperature=temperature,
        max_tokens=16384,
        request_timeout=240.0,
        extra_body=extra_body,
    )
    embeddings = OpenAIEmbeddings(
        model=embed_model,
        api_key=api_key,
        base_url=base_url,
        check_embedding_ctx_length=False,
        model_kwargs={"encoding_format": "float"},
    )

    llm_wrapper = (
        LangchainLLMWrapper(llm, is_finished_parser=lambda _: True)
        if is_generator
        else LangchainLLMWrapper(llm)
    )
    return llm_wrapper, LangchainEmbeddingsWrapper(embeddings)
# endregion

## Step 1: Synthetic Test Dataset Generation (Optional)

Generate diverse, multi-hop question-answer test sets directly from document chunks using Ragas `TestsetGenerator` and Knowledge Graph extraction.

> **Note**: This step makes calls to OpenRouter. If you already have `1.eval_dataset.json`, you can skip directly to Step 2.

In [ ]:
# region Generator
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.synthesizers.multi_hop import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)
from ragas.testset.transforms.engine import Parallel
from ragas.testset.transforms.extractors import EmbeddingExtractor, SummaryExtractor
from ragas.testset.transforms.extractors.llm_based import NERExtractor, ThemesExtractor
from ragas.testset.transforms.filters import CustomNodeFilter
from ragas.testset.transforms.relationship_builders import (
    CosineSimilarityBuilder,
    OverlapScoreBuilder,
)


def load_input_documents(doc_path: Path, max_chunks: int = 20) -> list[Document]:
    """Loads documents from text files or sampled JSON chunks."""
    if not doc_path.exists():
        raise FileNotFoundError(f"[DatasetGenerator-load] Target document not found: {doc_path}")

    if doc_path.suffix.lower() == ".json":
        items = json.loads(doc_path.read_text(encoding="utf-8"))
        items = items if isinstance(items, list) else [items]
        docs = []
        for item in items:
            meta = item.get("metadata", {})
            content = (
                meta.get("big") or meta.get("summary") or item.get("small") or ""
            ).strip()
            if len(content) > 50:
                docs.append(Document(page_content=content, metadata={"id": item.get("id", "")}))
            if 0 < max_chunks <= len(docs):
                break
        return docs

    return TextLoader(str(doc_path), encoding="utf-8").load()


def build_query_distribution(ragas_llm, weights: dict[str, float]) -> list[tuple]:
    """Builds a normalized 4-path query distribution matrix."""
    synthesizers = {
        "single_specific": SingleHopSpecificQuerySynthesizer(
            llm=ragas_llm, property_name="entities", name="single_hop_specific"
        ),
        "single_abstract": SingleHopSpecificQuerySynthesizer(
            llm=ragas_llm, property_name="themes", name="single_hop_abstract"
        ),
        "multi_specific": MultiHopSpecificQuerySynthesizer(
            llm=ragas_llm, name="multi_hop_specific"
        ),
        "multi_abstract": MultiHopAbstractQuerySynthesizer(
            llm=ragas_llm, name="multi_hop_abstract"
        ),
    }
    raw = [(synthesizers[k], max(0.0, weights.get(k, 0.25))) for k in synthesizers]
    total = sum(w for _, w in raw) or 1.0
    return [(s, w / total) for s, w in raw if w > 0]


def generate_eval_dataset_from_docs(
    doc_path: Path,
    output_path: Path,
    test_size: int = 30,
    max_chunks: int = 20,
    weights: dict[str, float] | None = None,
) -> list[dict]:
    """Synthesizes evaluation test samples in LangSmith schema with ground_truth outputs."""
    print(f"[DatasetGenerator-load] Loading up to {max_chunks} chunks from: {doc_path.name}")
    docs = load_input_documents(doc_path, max_chunks=max_chunks)
    if not docs:
        raise ValueError(f"[DatasetGenerator-load] No readable document content found in {doc_path}")

    ragas_llm, ragas_embeddings = get_eval_models(is_generator=True)
    transforms = [
        SummaryExtractor(llm=ragas_llm),
        CustomNodeFilter(llm=ragas_llm),
        Parallel(
            EmbeddingExtractor(
                embedding_model=ragas_embeddings,
                property_name="summary_embedding",
                embed_property_name="summary",
            ),
            ThemesExtractor(llm=ragas_llm),
            NERExtractor(llm=ragas_llm),
        ),
        Parallel(
            CosineSimilarityBuilder(
                property_name="summary_embedding",
                new_property_name="summary_similarity",
                threshold=0.1,
            ),
            OverlapScoreBuilder(threshold=0.01),
        ),
    ]

    distribution = build_query_distribution(ragas_llm, weights or {})
    summary_str = ", ".join(f"{s.name}: {w:.0%}" for s, w in distribution)
    print(f"[DatasetGenerator-generate] Synthesizing {test_size} samples ({summary_str}) across {len(docs)} chunks...")

    generator = TestsetGenerator(
        llm=ragas_llm,
        embedding_model=ragas_embeddings,
        knowledge_graph=KnowledgeGraph(),
    )
    run_config = RunConfig(max_workers=4, max_retries=5, timeout=240)
    dataset = generator.generate_with_langchain_docs(
        documents=docs,
        testset_size=test_size,
        transforms=transforms,
        query_distribution=distribution,
        run_config=run_config,
    )

    # Clean schema: inputs (query) and outputs (ground_truth only)
    samples = [
        {
            "inputs": {
                "query": row.get("user_input") or row.get("question") or "",
            },
            "outputs": {
                "ground_truth": row.get("reference") or row.get("ground_truth") or "",
            },
            "metadata": {
                "id": f"sample-{idx + 1:02d}",
                "ground_truth_contexts": list(row.get("reference_contexts") or []),
                "synthesizer": row.get("synthesizer_name") or "",
            },
        }
        for idx, row in dataset.to_pandas().iterrows()
    ]

    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(samples, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[DatasetGenerator-save] Saved {len(samples)} samples to {output_path.name}")
    return samples
# endregion

In [ ]:
# region ExecGenerator
# Set EXECUTE_GENERATOR to True to trigger OpenRouter testset generation
EXECUTE_GENERATOR = False

if EXECUTE_GENERATOR:
    raw_chunks_file = CURRENT_DIR / "0.chunks.json"
    target_dataset_file = CURRENT_DIR / "1.eval_dataset.json"
    
    generated_samples = generate_eval_dataset_from_docs(
        doc_path=raw_chunks_file,
        output_path=target_dataset_file,
        test_size=30,
        max_chunks=20,
    )
else:
    print("[DatasetGenerator-run] Generation skipped. Set EXECUTE_GENERATOR = True to re-synthesize test set.")
# endregion

## Step 2: Inspect Evaluation Dataset (LangSmith Schema)

Review the evaluation dataset formatted in the standard LangSmith schema:
- **`inputs`**: `{"query": "..."}` matching LangGraph's `InputState`.
- **`outputs`**: `{"ground_truth": "..."}` containing the verified reference answer.
- **`metadata`**: `{"id": "...", "ground_truth_contexts": [...], "synthesizer": "..."}` for auditability.

In [ ]:
# region Inspection
dataset_file = CURRENT_DIR / "1.eval_dataset.json"

if dataset_file.exists():
    raw_data = json.loads(dataset_file.read_text(encoding="utf-8"))
    print(f"[Inspection-load] Loaded {len(raw_data)} samples from {dataset_file.name}\n")
    
    # Analyze synthesizer distribution
    synthesizers = [s.get("metadata", {}).get("synthesizer", "unknown") for s in raw_data]
    dist_df = pd.Series(synthesizers).value_counts().rename_axis("Synthesizer").reset_index(name="Count")
    print("### Query Distribution Breakdown:")
    print(dist_df.to_string(index=False))
    print()
    
    # Preview sample records in LangSmith schema
    preview_rows = []
    for item in raw_data[:5]:
        query = item.get("inputs", {}).get("query") or item.get("question", "")
        answer = item.get("outputs", {}).get("ground_truth") or item.get("right_answer", "")
        preview_rows.append({
            "ID": item.get("metadata", {}).get("id", ""),
            "Synthesizer": item.get("metadata", {}).get("synthesizer", ""),
            "Input Query": query[:80] + "...",
            "Ground Truth": answer[:80] + "...",
        })
    print("### Sample Records Preview:")
    display_df = pd.DataFrame(preview_rows)
    print(display_df.to_string(index=False))
else:
    print(f"[Inspection-load] Dataset file not found at: {dataset_file}")
# endregion

## Step 3: Query LangGraph Agent (Direct or Server)

Query the LangGraph RAG agent to collect generated responses and retrieved document contexts.

### Execution Modes:
- **`direct` (Recommended)**: Imports `agent_graph` from `src.agent_flow.graph` and executes directly in-process via `ainvoke()`. Fast, self-contained, no external server required.
- **`http`**: Sends requests to a running LangGraph Server (`http://localhost:2024/runs/wait`) via HTTP.

In [ ]:
# region Fetcher
async def fetch_single_langgraph_response(
    query: str,
    mode: str = "direct",
    endpoint_url: str = "http://localhost:2024",
    client: httpx.AsyncClient | None = None,
) -> tuple[str, list[str]]:
    """Fetches answer and retrieved contexts from LangGraph agent."""
    if mode == "direct":
        from src.agent_flow.graph import agent_graph

        state = await agent_graph.ainvoke({"query": query})
        answer = state.get("draft_response") or ""
        # Extract retrieved / ranked docs from LangGraph AgentState
        docs = state.get("ranked_docs") or state.get("retrieved_docs") or []
        contexts = [
            doc.page_content if hasattr(doc, "page_content") else str(doc)
            for doc in docs
        ]
        return answer, contexts

    elif mode == "http":
        if client is None:
            raise ValueError("[EvalRunner-fetch] AsyncClient must be provided for HTTP mode.")
        url = f"{endpoint_url.rstrip('/')}/runs/wait"
        payload = {"assistant_id": "agent", "input": {"query": query}}
        resp = await client.post(url, json=payload)
        resp.raise_for_status()
        data = resp.json()
        answer = data.get("draft_response") or ""
        raw_docs = data.get("ranked_docs") or data.get("retrieved_docs") or []
        contexts = [
            d.get("page_content", "") if isinstance(d, dict) else str(d)
            for d in raw_docs
        ]
        return answer, contexts

    else:
        raise ValueError(f"Unknown mode: {mode}")


async def fetch_all_langgraph_responses(
    dataset_path: Path,
    mode: str = "direct",
    endpoint_url: str = "http://localhost:2024",
    max_samples: int | None = None,
) -> list[dict]:
    """Queries the LangGraph agent for each test sample in dataset."""
    if not dataset_path.exists():
        raise FileNotFoundError(f"[EvalRunner-fetch] Dataset file not found: {dataset_path}")

    dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
    if max_samples:
        dataset = dataset[:max_samples]

    print(f"[EvalRunner-fetch] Running LangGraph ({mode} mode) across {len(dataset)} questions...")
    samples = []

    async with httpx.AsyncClient(timeout=90.0) as client:
        for idx, item in enumerate(dataset, 1):
            q = (item.get("inputs", {}).get("query") or item.get("question", "")).strip()
            if not q:
                continue

            print(f"[EvalRunner-fetch] [{idx:02d}/{len(dataset)}] Query: '{q[:60]}...'")
            answer, contexts = "Error executing query.", ["Error"]

            for attempt in range(1, 4):
                try:
                    answer, contexts = await fetch_single_langgraph_response(
                        query=q,
                        mode=mode,
                        endpoint_url=endpoint_url,
                        client=client if mode == "http" else None,
                    )
                    break
                except Exception as err:
                    print(f"[EvalRunner-fetch] Attempt {attempt}/3 failed for '{q[:40]}': {err}")
                    if attempt < 3:
                        await asyncio.sleep(2.0 * attempt)

            meta = item.get("metadata", {})
            gt_contexts = meta.get("ground_truth_contexts") or []
            expected_answer = (
                item.get("outputs", {}).get("ground_truth")
                or item.get("outputs", {}).get("draft_response")
                or item.get("right_answer", "")
            )

            samples.append(
                {
                    "question": q,
                    "answer": answer,
                    "contexts": contexts,
                    "right_answer": expected_answer,
                    "reference_contexts": gt_contexts,
                }
            )

    return samples
# endregion

In [ ]:
# region ExecFetch
dataset_file = CURRENT_DIR / "1.eval_dataset.json"

# Execution configuration:
EXECUTE_QUERY = False
QUERY_MODE = "direct"  # Options: "direct" (in-process, recommended) or "http" (port 2024)
LANGGRAPH_ENDPOINT = os.getenv("LANGGRAPH_ENDPOINT", "http://localhost:2024")
MAX_SAMPLES = None  # Set to an integer (e.g. 5) for quick sanity checks

rag_samples = []

if EXECUTE_QUERY:
    # In Jupyter, top-level await can also be used directly:
    # rag_samples = await fetch_all_langgraph_responses(dataset_file, mode=QUERY_MODE)
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

    if loop.is_running():
        import nest_asyncio
        nest_asyncio.apply()
        rag_samples = loop.run_until_complete(
            fetch_all_langgraph_responses(
                dataset_path=dataset_file,
                mode=QUERY_MODE,
                endpoint_url=LANGGRAPH_ENDPOINT,
                max_samples=MAX_SAMPLES,
            )
        )
    else:
        rag_samples = loop.run_until_complete(
            fetch_all_langgraph_responses(
                dataset_path=dataset_file,
                mode=QUERY_MODE,
                endpoint_url=LANGGRAPH_ENDPOINT,
                max_samples=MAX_SAMPLES,
            )
        )
    print(f"[EvalRunner-fetch] Collected {len(rag_samples)} responses from LangGraph agent.")
else:
    print("[EvalRunner-fetch] Query execution skipped. Set EXECUTE_QUERY = True to run queries.")
# endregion

## Step 4: Compute RAGAS Metrics

Evaluate retrieved contexts and answers against ground truth references using RAGAS:
- **Faithfulness**: Is the response grounded in the retrieved context? (Target: `>= 0.80`)
- **Answer Relevancy**: Is the response directly addressing the user inquiry? (Target: `>= 0.75`)
- **Context Precision**: How high is the ranking of relevant chunks in retrieval? (Target: `>= 0.85`)
- **Context Recall**: Did retrieval cover all facts from the ground truth answer? (Target: `>= 0.80`)

In [ ]:
# region Evaluation
def run_ragas_evaluation(samples: list[dict], output_dir: Path) -> pd.DataFrame:
    """Executes RAGAS evaluation on collected samples and exports results to CSV."""
    if not samples:
        raise ValueError("[EvalRunner-eval] No samples provided for evaluation.")

    eval_llm, eval_embeddings = get_eval_models()
    eval_dataset = Dataset.from_dict(
        {
            "question": [s["question"] for s in samples],
            "answer": [s["answer"] for s in samples],
            "contexts": [s["contexts"] for s in samples],
            "ground_truth": [s["right_answer"] for s in samples],
            "reference_contexts": [list(s.get("reference_contexts") or []) for s in samples],
        }
    )

    metrics = [faithfulness, answer_relevancy, context_precision, context_recall]
    print(f"[EvalRunner-eval] Evaluating {len(samples)} samples with RAGAS metrics...")
    
    result = evaluate(
        dataset=eval_dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
        run_config=RunConfig(max_workers=2, max_retries=5, timeout=240),
    )

    df: pd.DataFrame = result.to_pandas()
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = output_dir / f"2.run_eval_{timestamp}.csv"

    avg_row = {col: "-" for col in df.columns}
    avg_row["user_input"] = "[AVERAGE SUMMARY]"
    for m in metrics:
        if m.name in df.columns:
            avg_row[m.name] = round(float(df[m.name].mean()), 4)

    df_out = pd.concat([pd.DataFrame([avg_row]), df], ignore_index=True)
    df_out.to_csv(csv_path, index=False)

    print(f"\n[EvalRunner-save] CSV saved to: {csv_path}")
    print("\n### Mean Evaluation Scores:")
    for m in metrics:
        if m.name in avg_row:
            print(f"- {m.name:20s}: {avg_row[m.name]}")

    return df_out
# endregion

In [ ]:
# region ExecEval
# Set EXECUTE_EVAL to True to run RAGAS benchmark on collected samples
EXECUTE_EVAL = False

if EXECUTE_EVAL and rag_samples:
    eval_results_df = run_ragas_evaluation(samples=rag_samples, output_dir=CURRENT_DIR)
elif EXECUTE_EVAL and not rag_samples:
    print("[EvalRunner-eval] Cannot run evaluation: rag_samples is empty. Run Step 3 first.")
else:
    print("[EvalRunner-eval] Evaluation execution skipped. Set EXECUTE_EVAL = True to execute.")
# endregion

## Step 5: Historical Evolution & Visualization

Load historical benchmark run CSVs (`2.run_eval_*.csv`), compare iterations against target quality thresholds, and plot the metric trajectories.

In [ ]:
# region Visualization
def load_benchmark_history(eval_dir: Path) -> pd.DataFrame:
    """Parses all historical evaluation CSV files and extracts summary rows."""
    csv_files = sorted(eval_dir.glob("2.run_eval_*.csv"))
    if not csv_files:
        print(f"[Visualization-load] No CSV benchmark files found in {eval_dir}")
        return pd.DataFrame()

    records = []
    metric_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

    for file_path in csv_files:
        # Extract timestamp from filename: 2.run_eval_YYYYMMDD_HHMMSS.csv
        match = re.search(r"2\.run_eval_(\d{8})_(\d{6})\.csv", file_path.name)
        if match:
            date_str = f"{match.group(1)[:4]}-{match.group(1)[4:6]}-{match.group(1)[6:]}"
            time_str = f"{match.group(2)[:2]}:{match.group(2)[2:4]}:{match.group(2)[4:]}"
            label = f"{date_str} {time_str}"
        else:
            label = file_path.stem.replace("2.run_eval_", "")

        try:
            df = pd.read_csv(file_path)
            # Find average summary row or compute mean
            summary_mask = df["user_input"].astype(str).str.contains("AVERAGE", case=False, na=False)
            if summary_mask.any():
                summary_row = df[summary_mask].iloc[0]
            else:
                summary_row = df[metric_cols].mean()

            entry = {"Run": label, "File": file_path.name}
            for col in metric_cols:
                val = summary_row.get(col, None)
                try:
                    entry[col] = round(float(val), 4)
                except (ValueError, TypeError):
                    entry[col] = None
            records.append(entry)
        except Exception as err:
            print(f"[Visualization-load] Error parsing {file_path.name}: {err}")

    history_df = pd.DataFrame(records)
    return history_df


def plot_benchmark_trends(history_df: pd.DataFrame):
    """Renders multi-line trajectory charts across historical benchmark runs."""
    if history_df.empty:
        print("[Visualization-plot] No benchmark data available to plot.")
        return

    metric_targets = {
        "faithfulness": 0.80,
        "answer_relevancy": 0.75,
        "context_precision": 0.85,
        "context_recall": 0.80,
    }
    metric_colors = {
        "faithfulness": "#2563eb",       # Blue
        "answer_relevancy": "#059669",   # Green
        "context_precision": "#d97706",  # Amber
        "context_recall": "#9333ea",     # Purple
    }

    fig, ax = plt.subplots(figsize=(11, 5.5), dpi=100)
    x_labels = history_df["Run"].tolist()
    x_indices = list(range(len(x_labels)))

    for metric, target in metric_targets.items():
        if metric in history_df.columns:
            values = history_df[metric].tolist()
            color = metric_colors.get(metric, "#475569")
            ax.plot(x_indices, values, marker="o", linewidth=2.2, label=f"{metric.replace('_', ' ').title()} (Target: {target:.2f})", color=color)
            # Reference target line
            ax.axhline(y=target, color=color, linestyle=":", alpha=0.4)

    ax.set_title("RAG Metric Progression Across Benchmark Runs", fontsize=14, fontweight="bold", pad=14)
    ax.set_xlabel("Benchmark Run Timestamp", fontsize=11, labelpad=10)
    ax.set_ylabel("Score (0.0 - 1.0)", fontsize=11, labelpad=10)
    ax.set_xticks(x_indices)
    ax.set_xticklabels(x_labels, rotation=25, ha="right", fontsize=9)
    ax.set_ylim(0.2, 1.05)
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="lower right", framealpha=0.9, fontsize=9)
    plt.tight_layout()
    plt.show()


history_df = load_benchmark_history(CURRENT_DIR)
print("### Historical Benchmark Summary Table:")
print(history_df.to_string(index=False))
print()
plot_benchmark_trends(history_df)
# endregion

## Step 6: Synchronize with Evaluation Journal

Automatically format the latest evaluation result as a Markdown table row and append it to `JOURNAL.md`.

In [ ]:
# region Journal
def sync_latest_eval_to_journal(
    eval_dir: Path,
    milestone_name: str,
    journal_path: Path | None = None,
) -> bool:
    """Appends the latest evaluation CSV summary row to JOURNAL.md (or EVALUATION_JOURNAL.md)."""
    if journal_path is None:
        primary = eval_dir / "JOURNAL.md"
        legacy = eval_dir / "EVALUATION_JOURNAL.md"
        journal_path = primary if primary.exists() else legacy

    if not journal_path.exists():
        print(f"[Journal-sync] Journal file not found: {journal_path}")
        return False

    csv_files = sorted(eval_dir.glob("2.run_eval_*.csv"))
    if not csv_files:
        print(f"[Journal-sync] No evaluation CSV files found in {eval_dir}")
        return False

    latest_csv = csv_files[-1]
    match = re.search(r"2\.run_eval_(\d{8})_(\d{6})\.csv", latest_csv.name)
    if not match:
        print(f"[Journal-sync] Latest file format unexpected: {latest_csv.name}")
        return False

    date_str = f"{match.group(1)[:4]}-{match.group(1)[4:6]}-{match.group(1)[6:]}"
    time_str = f"{match.group(2)[:2]}:{match.group(2)[2:4]}:{match.group(2)[4:]}"

    df = pd.read_csv(latest_csv)
    summary_mask = df["user_input"].astype(str).str.contains("AVERAGE", case=False, na=False)
    summary_row = df[summary_mask].iloc[0] if summary_mask.any() else df[["faithfulness", "answer_relevancy", "context_precision", "context_recall"]].mean()

    f = float(summary_row.get("faithfulness", 0.0))
    ar = float(summary_row.get("answer_relevancy", 0.0))
    cp = float(summary_row.get("context_precision", 0.0))
    cr = float(summary_row.get("context_recall", 0.0))

    new_row = f"| **{date_str}** (`{time_str}`) | {milestone_name} | `{f:.4f}` | `{ar:.4f}` | `{cp:.4f}` | `{cr:.4f}` |"
    print(f"[Journal-sync] Generated Row:\n{new_row}")

    content = journal_path.read_text(encoding="utf-8")
    lines = content.splitlines()

    table_indices = [idx for idx, line in enumerate(lines) if line.strip().startswith("|") and ("Faithfulness" in line or "`" in line or ":---" in line)]
    if not table_indices:
        print("[Journal-sync] Could not locate benchmark summary table in journal.")
        return False

    last_table_idx = max(table_indices)
    lines.insert(last_table_idx + 1, new_row)
    journal_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"[Journal-sync] Successfully appended latest run to {journal_path.name}")
    return True


# Set EXECUTE_JOURNAL_SYNC to True and provide milestone title to append latest CSV run
EXECUTE_JOURNAL_SYNC = False
MILESTONE_TITLE = "[LangGraph Agent] Native Graph Benchmark Run"

if EXECUTE_JOURNAL_SYNC:
    sync_latest_eval_to_journal(
        eval_dir=CURRENT_DIR,
        milestone_name=MILESTONE_TITLE,
    )
else:
    print("[Journal-sync] Sync skipped. Set EXECUTE_JOURNAL_SYNC = True when ready to log run.")
# endregion

## Step 7: Upload Dataset to LangSmith

Upload and synchronize `1.eval_dataset.json` directly into LangSmith as an evaluation dataset.

Requires:
- `LANGSMITH_API_KEY`: Set in `.env`
- `LANGSMITH_PROJECT` / `LANGSMITH_ENDPOINT`: Configured in `.env`

In [ ]:
# region LangSmith
from langsmith import Client

def upload_dataset_to_langsmith(
    dataset_path: Path,
    dataset_name: str = "kanzi-documentation-eval",
    description: str = "Kanzi QA RAG evaluation test set synthesized via KnowledgeGraph",
) -> str | None:
    """Uploads local JSON dataset to LangSmith with inputs, outputs, and metadata."""
    api_key = os.getenv("LANGSMITH_API_KEY")
    if not api_key or api_key == "mock_key":
        print("[LangSmith-sync] LANGSMITH_API_KEY is not configured or set to mock_key.")
        return None

    if not dataset_path.exists():
        raise FileNotFoundError(f"[LangSmith-sync] Dataset file not found: {dataset_path}")

    raw_samples = json.loads(dataset_path.read_text(encoding="utf-8"))
    client = Client()

    # Check if dataset already exists
    if client.has_dataset(dataset_name=dataset_name):
        dataset = client.read_dataset(dataset_name=dataset_name)
        print(f"[LangSmith-sync] Found existing dataset '{dataset_name}' (ID: {dataset.id})")
    else:
        dataset = client.create_dataset(
            dataset_name=dataset_name,
            description=description,
        )
        print(f"[LangSmith-sync] Created new dataset '{dataset_name}' (ID: {dataset.id})")

    inputs = [s.get("inputs", {"query": s.get("question", "")}) for s in raw_samples]
    outputs = [s.get("outputs", {"ground_truth": s.get("right_answer", "")}) for s in raw_samples]
    metadata = [s.get("metadata", {}) for s in raw_samples]

    print(f"[LangSmith-sync] Ingesting {len(raw_samples)} examples into LangSmith...")
    client.create_examples(
        inputs=inputs,
        outputs=outputs,
        metadata=metadata,
        dataset_id=dataset.id,
    )
    print(f"[LangSmith-sync] Successfully uploaded {len(raw_samples)} examples to LangSmith dataset '{dataset_name}'!")
    return str(dataset.id)


# Set EXECUTE_LANGSMITH_SYNC = True when you wish to inject dataset to LangSmith
EXECUTE_LANGSMITH_SYNC = False
LANGSMITH_DATASET_NAME = os.getenv("LANGSMITH_DATASET_NAME", "kanzi-documentation-eval")

if EXECUTE_LANGSMITH_SYNC:
    dataset_file = CURRENT_DIR / "1.eval_dataset.json"
    upload_dataset_to_langsmith(
        dataset_path=dataset_file,
        dataset_name=LANGSMITH_DATASET_NAME,
    )
else:
    print("[LangSmith-sync] Ingestion skipped. Set EXECUTE_LANGSMITH_SYNC = True to upload to LangSmith.")
# endregion

## Step 8: All-in-One Automated Benchmark Execution

Execute the full RAG quality evaluation pipeline in a single step (query LangGraph -> run Ragas scoring -> sync to journal).

In [ ]:
# region AllInOne
async def run_full_pipeline(
    mode: str = "direct",
    milestone_name: str = "[LangGraph Agent] Automated All-in-One Evaluation",
    max_samples: int | None = None,
    sync_langsmith: bool = False,
):
    """Executes full evaluation sequence end-to-end."""
    dataset_file = CURRENT_DIR / "1.eval_dataset.json"
    print(f"[AllInOne-start] Starting full pipeline (mode={mode})...")

    # 1. Fetch responses from LangGraph agent
    samples = await fetch_all_langgraph_responses(
        dataset_path=dataset_file,
        mode=mode,
        max_samples=max_samples,
    )

    # 2. Evaluate with Ragas
    eval_df = run_ragas_evaluation(samples=samples, output_dir=CURRENT_DIR)

    # 3. Sync summary into JOURNAL.md
    sync_latest_eval_to_journal(
        eval_dir=CURRENT_DIR,
        milestone_name=milestone_name,
    )

    # 4. Optionally sync dataset to LangSmith
    if sync_langsmith:
        upload_dataset_to_langsmith(dataset_file)

    # 5. Refresh historical plots
    updated_history_df = load_benchmark_history(CURRENT_DIR)
    plot_benchmark_trends(updated_history_df)
    print("[AllInOne-complete] Pipeline completed successfully!")


# Set to True to trigger full evaluation end-to-end
EXECUTE_ALL_IN_ONE = False

if EXECUTE_ALL_IN_ONE:
    # In Jupyter: await run_full_pipeline(mode="direct")
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

    if loop.is_running():
        import nest_asyncio
        nest_asyncio.apply()
        loop.run_until_complete(run_full_pipeline(mode="direct", max_samples=None))
    else:
        loop.run_until_complete(run_full_pipeline(mode="direct", max_samples=None))
else:
    print("[AllInOne-run] Skipped. Set EXECUTE_ALL_IN_ONE = True to run full pipeline.")
# endregion